In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load the regression dataset
file_path = "Input_Data/01_cleaned_data/uwb_dataset_regression.parquet"
df = pd.read_parquet(file_path)

# 2. Separate features (X) and target (y)
target_col = "TARGET_Second_Path_Range"
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found.")

X = df.drop(columns=[target_col, 'NLOS'], errors='ignore')
y = df[target_col]

# 3. Split into 80% Training and 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dataset loaded. Training on {X_train.shape[0]} rows and {X_train.shape[1]} features.")

# Store model-level metrics for final summary
model_results = []

# 4. Regression Dashboard Plotting Function
def plot_independent_dashboard(model_name, grid, param_name):
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    cv_res = pd.DataFrame(grid.cv_results_)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    model_results.append({
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Best Params': grid.best_params_
    })

    residuals = y_test - y_pred

    fig, axes = plt.subplots(2, 2, figsize=(26, 5))
    fig.suptitle(f"Model Results: {model_name} (Regression)", fontsize=18, fontweight='bold')

    # Graph 1: Train vs CV RMSE across grid values
    param_values = [str(x) for x in cv_res[f'param_{param_name}']]
    train_rmse = -cv_res['mean_train_score']
    test_rmse = -cv_res['mean_test_score']
    axes[0].plot(param_values, train_rmse, marker='s', label='Train RMSE', color='skyblue', lw=2)
    axes[0].plot(param_values, test_rmse, marker='o', label='CV RMSE', color='salmon', lw=2)
    axes[0].set_title("Learning Curve: Train vs. CV RMSE")
    axes[0].set_xlabel(param_name)
    axes[0].set_ylabel("RMSE (lower is better)")
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=30)

    # Graph 2: Predicted vs Actual
    axes[1].scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    axes[1].set_title("Predicted vs Actual")
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predicted")
    axes[1].legend()

    # Graph 3: Residual distribution
    sns.histplot(residuals, bins=40, kde=True, ax=axes[2], color='darkorange')
    axes[2].axvline(0, color='black', linestyle='--', linewidth=1.2)
    axes[2].set_title("Residual Distribution")
    axes[2].set_xlabel("Residual (Actual - Predicted)")

    # Graph 4: Residuals vs Predicted (scatter)
    axes[3].scatter(y_pred, residuals, alpha=0.3, s=10, color='purple')
    axes[3].axhline(0, color='black', linestyle='--', linewidth=1.2)
    axes[3].set_title("Residuals vs Predicted")
    axes[3].set_xlabel("Predicted")
    axes[3].set_ylabel("Residual")

    plt.tight_layout()
    plt.show()

    print(f"Best Params: {grid.best_params_}")
    print(f"Test RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

In [ ]:
from sklearn.linear_model import Ridge
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(
    Ridge(), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train)
plot_independent_dashboard("Ridge Regression", grid, 'alpha')

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
param_grid = {'n_neighbors': [3, 7, 15]}
grid = GridSearchCV(
    KNeighborsRegressor(), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train)
plot_independent_dashboard("K-Nearest Neighbors Regressor", grid, 'n_neighbors')

In [ ]:
from sklearn.tree import DecisionTreeRegressor
param_grid = {'max_depth': [5, 15, 30, None]}
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train)
plot_independent_dashboard("Decision Tree Regressor", grid, 'max_depth')

In [ ]:
from sklearn.ensemble import RandomForestRegressor
param_grid = {'n_estimators': [50, 100, 200]}
grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train)
plot_independent_dashboard("Random Forest Regressor", grid, 'n_estimators')

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
param_grid = {'n_estimators': [50, 100, 200]}
grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train)
plot_independent_dashboard("Gradient Boosting Regressor", grid, 'n_estimators')

In [ ]:
from sklearn.svm import SVR
param_grid = {'C': [0.1, 1, 10]}
grid = GridSearchCV(
    SVR(kernel='rbf'), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train)
plot_independent_dashboard("Support Vector Regressor", grid, 'C')

In [ ]:
from sklearn.neural_network import MLPRegressor
param_grid = {'alpha': [0.0001, 0.01, 0.1]}
grid = GridSearchCV(
    MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
    param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train)
plot_independent_dashboard("Neural Network Regressor", grid, 'alpha')

In [ ]:
# Final summary across all tuned regression models
summary_df = pd.DataFrame(model_results).copy()
summary_df['RMSE Rank'] = summary_df['RMSE'].rank(method='min', ascending=True)
summary_df['MAE Rank'] = summary_df['MAE'].rank(method='min', ascending=True)
summary_df['R2 Rank'] = summary_df['R2'].rank(method='min', ascending=False)
summary_df['Overall Rank'] = summary_df[['RMSE Rank', 'MAE Rank', 'R2 Rank']].sum(axis=1)
summary_df = summary_df.sort_values('Overall Rank').reset_index(drop=True)
display(summary_df[['Model', 'RMSE', 'MAE', 'R2', 'Best Params', 'Overall Rank']])

fig, axes = plt.subplots(3, 1, figsize=(14, 18))
fig.suptitle('Final Model Comparison', fontsize=16, fontweight='bold')

# Graph 1: RMSE
rmse_colors = ['#2ecc71' if v == summary_df['RMSE'].min() else '#3498db' for v in summary_df['RMSE']]
axes[0].bar(summary_df['Model'], summary_df['RMSE'], color=rmse_colors, edgecolor='black')
axes[0].set_title('RMSE by Model (lower is better)')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Graph 2: MAE
mae_colors = ['#2ecc71' if v == summary_df['MAE'].min() else '#3498db' for v in summary_df['MAE']]
axes[1].bar(summary_df['Model'], summary_df['MAE'], color=mae_colors, edgecolor='black')
axes[1].set_title('MAE by Model (lower is better)')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

# Graph 3: R2
r2_colors = ['#2ecc71' if v == summary_df['R2'].max() else '#3498db' for v in summary_df['R2']]
axes[2].bar(summary_df['Model'], summary_df['R2'], color=r2_colors, edgecolor='black')
axes[2].set_title('R2 by Model (higher is better)')
axes[2].set_ylabel('R2')
axes[2].set_xlabel('Model')
axes[2].tick_params(axis='x', rotation=30)
axes[2].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

best_overall = summary_df.iloc[0]
best_rmse_model = summary_df.loc[summary_df['RMSE'].idxmin(), 'Model']
best_mae_model = summary_df.loc[summary_df['MAE'].idxmin(), 'Model']
best_r2_model = summary_df.loc[summary_df['R2'].idxmax(), 'Model']

print("Best model by combined ranking:", best_overall['Model'])
print(f"Overall Rank Score: {best_overall['Overall Rank']:.0f}")
print(f"Best RMSE model: {best_rmse_model}")
print(f"Best MAE model : {best_mae_model}")
print(f"Best R2 model  : {best_r2_model}")
print("\nConclusion: the best overall model is the one with the lowest combined rank across RMSE, MAE, and R2.")